In [16]:
# Verify exported dataset

exported_df = pd.read_csv(output_path)

print("Exported file shape:", exported_df.shape)
print("Exported file exists:", output_path.exists())
print("Duplicate CustomerIDs:", exported_df["CustomerID"].duplicated().sum())

Exported file shape: (7043, 37)
Exported file exists: True
Duplicate CustomerIDs: 0


In [15]:
# STEP 14 — Export Clean Dataset

output_path = project_root / "05_Python" / "outputs" / "cleaned_data.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset exported successfully.")
print("File:", output_path)
print("Shape:", df.shape)

Cleaned dataset exported successfully.
File: c:\Users\B ANAND\OneDrive\Desktop\project 1\telecom-customer-analytics\05_Python\outputs\cleaned_data.csv
Shape: (7043, 37)


In [14]:
# Final Pandas validation summary

validation_summary = {
    "Total Records": len(df),
    "Total Columns": 33,
    "Unique CustomerIDs": df["CustomerID"].nunique(),
    "Duplicate Rows": df.duplicated().sum(),
    "Duplicate CustomerIDs": df["CustomerID"].duplicated().sum(),
    "Churned Customers": (df["Churn Value"] == 1).sum(),
    "Active Customers": (df["Churn Value"] == 0).sum(),
    "Churn Rate (%)": round(
        (df["Churn Value"] == 1).sum() / len(df) * 100, 2
    ),
    "NULL Total Charges": df["Total Charges"].isna().sum(),
    "NULL Churn Flags": df["Churn Flag"].isna().sum(),
    "NULL Service Counts": df["Service Count"].isna().sum(),
    "NULL Tenure Groups": df["Tenure Group"].isna().sum(),
    "NULL Charge Bands": df["Monthly Charge Band"].isna().sum()
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation", "Result"]
)

validation_df

,Validation,Result
0,Total Records,7043.00
1,Total Columns,33.00
2,Unique CustomerIDs,7043.00
3,Duplicate Rows,0.00
4,Duplicate CustomerIDs,0.00
5,Churned Customers,1869.00
6,Active Customers,5174.00
7,Churn Rate (%),26.54
8,NULL Total Charges,11.00
9,NULL Churn Flags,0.00


In [13]:
# Validate Churn Flag logic

df["Churn Flag"] = df["Churn Label"].map({
    "Yes": 1,
    "No": 0
})

invalid_churn_flags = (
    df["Churn Flag"].isna() |
    (df["Churn Flag"] != df["Churn Value"])
).sum()

print("Invalid Churn Flags:", invalid_churn_flags)

print("\nChurn Flag counts:")
print(df["Churn Flag"].value_counts().sort_index())

Invalid Churn Flags: 0

Churn Flag counts:
Churn Flag
0    5174
1    1869
Name: count, dtype: int64


In [12]:
# Validate Service Count logic

df["Service Count"] = (
    (df["Phone Service"] == "Yes").astype(int)
    + df["Internet Service"].isin(["DSL", "Fiber optic"]).astype(int)
    + (df["Online Security"] == "Yes").astype(int)
    + (df["Online Backup"] == "Yes").astype(int)
    + (df["Device Protection"] == "Yes").astype(int)
    + (df["Tech Support"] == "Yes").astype(int)
    + (df["Streaming TV"] == "Yes").astype(int)
    + (df["Streaming Movies"] == "Yes").astype(int)
)

print("Minimum Service Count:", df["Service Count"].min())
print("Maximum Service Count:", df["Service Count"].max())
print("Average Service Count:", round(df["Service Count"].mean(), 4))
print("NULL Service Counts:", df["Service Count"].isna().sum())

Minimum Service Count: 1
Maximum Service Count: 8
Average Service Count: 3.7244
NULL Service Counts: 0


In [11]:
# Validate derived business logic

df["Tenure Group"] = pd.cut(
    df["Tenure Months"],
    bins=[-1, 12, 24, 48, float("inf")],
    labels=["0-12 Months", "13-24 Months", "25-48 Months", "49+ Months"]
)

df["Monthly Charge Band"] = pd.cut(
    df["Monthly Charges"],
    bins=[-float("inf"), 50.20, 84.15, float("inf")],
    labels=["Low", "Medium", "High"]
)

print("Tenure Group NULLs:", df["Tenure Group"].isna().sum())
print("Monthly Charge Band NULLs:", df["Monthly Charge Band"].isna().sum())

print("\nTenure Group counts:")
print(df["Tenure Group"].value_counts().sort_index())

print("\nMonthly Charge Band counts:")
print(df["Monthly Charge Band"].value_counts().sort_index())

Tenure Group NULLs: 0
Monthly Charge Band NULLs: 0

Tenure Group counts:
Tenure Group
0-12 Months     2186
13-24 Months    1024
25-48 Months    1594
49+ Months      2239
Name: count, dtype: int64

Monthly Charge Band counts:
Monthly Charge Band
Low       2327
Medium    2393
High      2323
Name: count, dtype: int64


In [10]:
# Validate Churn Reason logic

churned_without_reason = (
    (df["Churn Value"] == 1) &
    (df["Churn Reason"].isna())
).sum()

active_with_reason = (
    (df["Churn Value"] == 0) &
    (df["Churn Reason"].notna())
).sum()

print("Churned customers without Churn Reason:", churned_without_reason)
print("Active customers with Churn Reason:", active_with_reason)

Churned customers without Churn Reason: 0
Active customers with Churn Reason: 0


In [9]:
# Validate Total Charges NULL values

null_total_charges = df["Total Charges"].isna().sum()

zero_tenure_with_null_total = (
    df["Total Charges"].isna()
    & (df["Tenure Months"] == 0)
).sum()

print("NULL Total Charges:", null_total_charges)
print(
    "NULL Total Charges with zero tenure:",
    zero_tenure_with_null_total
)

NULL Total Charges: 11
NULL Total Charges with zero tenure: 11


In [8]:
# Numeric metric validation

# Convert Total Charges from text to numeric
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

print("MONTHLY CHARGES")
print("Minimum:", df["Monthly Charges"].min())
print("Maximum:", df["Monthly Charges"].max())
print("Mean:", round(df["Monthly Charges"].mean(), 2))

print("\nTOTAL CHARGES")
print("Minimum:", df["Total Charges"].min())
print("Maximum:", df["Total Charges"].max())
print("Mean:", round(df["Total Charges"].mean(), 2))

print("\nTENURE MONTHS")
print("Minimum:", df["Tenure Months"].min())
print("Maximum:", df["Tenure Months"].max())
print("Mean:", round(df["Tenure Months"].mean(), 2))

MONTHLY CHARGES
Minimum: 18.25
Maximum: 118.75
Mean: 64.76

TOTAL CHARGES
Minimum: 18.8
Maximum: 8684.8
Mean: 2283.3

TENURE MONTHS
Minimum: 0
Maximum: 72
Mean: 32.37


In [7]:
# Churn validation

churn_counts = df["Churn Value"].value_counts().sort_index()

print("Churn counts:")
print(churn_counts)

churned_customers = (df["Churn Value"] == 1).sum()
active_customers = (df["Churn Value"] == 0).sum()
total_customers = len(df)

churn_rate = churned_customers / total_customers * 100

print("\nChurned customers:", churned_customers)
print("Active customers:", active_customers)
print("Churn rate:", round(churn_rate, 2), "%")

Churn counts:
Churn Value
0    5174
1    1869
Name: count, dtype: int64

Churned customers: 1869
Active customers: 5174
Churn rate: 26.54 %


In [6]:
# Duplicate and CustomerID validation

duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["CustomerID"].duplicated().sum()
unique_customers = df["CustomerID"].nunique()

print("Duplicate rows:", duplicate_rows)
print("Duplicate CustomerIDs:", duplicate_customer_ids)
print("Unique CustomerIDs:", unique_customers)

Duplicate rows: 0
Duplicate CustomerIDs: 0
Unique CustomerIDs: 7043


In [5]:
# Missing-value validation

missing_values = df.isna().sum()

print("Columns with missing values:")
print(missing_values[missing_values > 0])

Columns with missing values:
Zip Code           1
Churn Reason    5174
dtype: int64


In [4]:
print("DataFrame shape:", df.shape)

print("\nDataFrame information:")
df.info()

DataFrame shape: (7043, 33)

DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7042 non-null   float64
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7

In [3]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parents[1]

csv_path = project_root / "03_Raw_Data" / "telecom_customer_churn.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully")
print("File:", csv_path)
print("Shape:", df.shape)

Dataset loaded successfully
File: c:\Users\B ANAND\OneDrive\Desktop\project 1\telecom-customer-analytics\03_Raw_Data\telecom_customer_churn.csv
Shape: (7043, 33)


# Phase 2 — Step 13: Pandas Cross-Validation

Purpose:
Independently validate the PostgreSQL ETL results using Pandas.

PostgreSQL remains the primary ETL and analytical source.
Pandas is used only for independent cross-validation.